In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from scipy.stats import wilcoxon
from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests


# Load CSV file
sheet_id = "1tEf8J1kbGZEFTOkoYqIjsdE3PkqBA9bX4MdNfMoLhjQ"
# tab_gid = "908803302"
# tab_gid = "1990888583" 
# tab_gid = "1868821623"
tab_gid = "1043571881"
full_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={tab_gid}"

In [49]:
import pandas as pd
from scipy.stats import wilcoxon

# Load CSV file
sheet_id = "1tEf8J1kbGZEFTOkoYqIjsdE3PkqBA9bX4MdNfMoLhjQ"
tab_gid = "1043571881"
full_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={tab_gid}"

In [72]:
def check_significance(df, col1, col2):
    stat, p_value = wilcoxon(df[col1], df[col2], alternative='greater')
    if p_value < 0.03:
        print(f"({col1} vs {col2}): {stat}, p-value: {p_value} \033[92mThe difference is statistically significant.\033[0m")
    else:
        print(f"({col1} vs {col2}): {stat}, p-value: {p_value} \033[91mThe difference is NOT statistically significant.\033[0m")

In [85]:
limit = 1001
suffix = ".4"
df = pd.read_csv(full_url, skiprows=1, usecols=[f"Original{suffix}", f"8-bit{suffix}", f"4-bit{suffix}", f"dyn{suffix}"]).iloc[:limit, :]
print(df.shape)
print(df.head(2))
check_significance(df, f"Original{suffix}", f"8-bit{suffix}")
check_significance(df, f"Original{suffix}", f"4-bit{suffix}")
check_significance(df, f"Original{suffix}", f"dyn{suffix}")

(1000, 4)
   Original.4  8-bit.4  4-bit.4  dyn.4
0       53.78    54.19    46.48  46.07
1       40.47    40.68    41.67  37.21
(Original.4 vs 8-bit.4): 149464.0, p-value: 0.03667882674540893 The difference is NOT statistically significant.
(Original.4 vs 4-bit.4): 265185.5, p-value: 1.2261890860399027e-06 The difference is statistically significant.
(Original.4 vs dyn.4): 364110.5, p-value: 1.627125662570729e-46 The difference is statistically significant.


## RQ1

In [68]:
limit = 1001
suffix = ".1"
df = pd.read_csv(full_url, skiprows=1, usecols=[f"Original{suffix}", f"8-bit{suffix}", f"4-bit{suffix}", f"dyn{suffix}"]).iloc[:limit, :]
print(df.shape)
print(df.head(2))
check_significance(df, f"Original{suffix}", f"8-bit{suffix}")
check_significance(df, f"Original{suffix}", f"4-bit{suffix}")
check_significance(df, f"Original{suffix}", f"dyn{suffix}")

(1000, 4)
   Original.1  8-bit.1  4-bit.1  dyn.1
0       38.36    46.07    34.19  35.44
1       38.78    41.39    39.51  34.75
(Original.1 vs 8-bit.1): 225077.5, p-value: 0.006333142597997131 The difference is statistically significant.
(Original.1 vs 4-bit.1): 348296.0, p-value: 3.748015854106487e-41 The difference is statistically significant.
(Original.1 vs dyn.1): 416936.5, p-value: 1.2004008010988297e-91 The difference is statistically significant.


### friedman test just for testing purpose

In [66]:
stat, p_value = friedmanchisquare(df['Original'], df['8-bit'], df['4-bit'])
print(f"Friedman chi² statistic: {stat:.3f}, p-value: {p_value:.4f}")

pairs = [("Original", "8-bit"), ("Original", "4-bit"), ("8-bit", "4-bit")]
p_values = [ wilcoxon(df[a], df[b])[1] for a, b in pairs]
rejected, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method="bonferroni")
for (a, b), p, corr_p in zip(pairs, p_values, corrected_p):
    print(f"{a} vs {b}: p={p:.4f}, corrected_p={corr_p:.4f}")

Friedman chi² statistic: 353.043, p-value: 0.0000
Original vs 8-bit: p=0.0000, corrected_p=0.0000
Original vs 4-bit: p=0.0000, corrected_p=0.0000
8-bit vs 4-bit: p=0.0000, corrected_p=0.0000


## RQ-2

In [4]:
colNames = ["ROC_AUC", "ROC_AUC.1", "ROC_AUC.2", "ROC_AUC.3"]
limit = 15
def check_roc_significance(df, score_col):
    df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", score_col]).iloc[:limit, :]
    df1 = df0.pivot(index="Model", columns="Compression", values=score_col)
    df = df1.reset_index()
    print(df.shape)
    check_significance(df, "Original", "8-bit")
    check_significance(df, "Original", "4-bit")
    

for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

colNames = ["PR_AUC", "PR_AUC.1", "PR_AUC.2", "PR_AUC.3"]
for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

Checking significance for ROC_AUC
(5, 4)
(Original vs 8-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for ROC_AUC.1
(5, 4)
(Original vs 8-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for ROC_AUC.2
(5, 4)
(Original vs 8-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for ROC_AUC.3
(5, 4)
(Original vs 8-bit): 5.0, p-value: 0.625 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for PR_AUC
(5, 4)
(Original vs 8-bit): 0.0, p-value: 0.06788915486182899 The diff

/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


(5, 4)
(Original vs 8-bit): 1.0, p-value: 0.125 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for PR_AUC.2
(5, 4)
(Original vs 8-bit): 1.0, p-value: 0.14412703481601533 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.
Checking significance for PR_AUC.3


/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


(5, 4)
(Original vs 8-bit): 2.0, p-value: 0.27332167829229814 The difference is NOT statistically significant.
(Original vs 4-bit): 0.0, p-value: 0.0625 The difference is NOT statistically significant.


/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


## RQ-4

### RQ-4.1: Task Performance

In [7]:
limit = 39
df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", "BLEU"]).iloc[:limit, :]
# print(df0.shape)
df1 = df0.pivot(index="Model", columns="Compression", values="BLEU")
df = df1.reset_index()
print(df.shape)
check_significance(df, "Original", "8-bit")
check_significance(df, "Original", "4-bit")

(11, 4)
(Original vs 8-bit): 44.5, p-value: 0.1826171875 The difference is NOT statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.


### RQ-4.2: Privacy Effectiveness

In [9]:
colNames = ["ROC_AUC", "ROC_AUC.1", "ROC_AUC.2", "ROC_AUC.3"]
limit = 39
def check_roc_significance(df, score_col):
    df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", score_col]).iloc[:limit, :]
    df1 = df0.pivot(index="Model", columns="Compression", values=score_col)
    df = df1.reset_index()
    print(df.shape)
    check_significance(df, "Original", "8-bit")
    check_significance(df, "Original", "4-bit")
    

for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

colNames = ["PR_AUC", "PR_AUC.1", "PR_AUC.2", "PR_AUC.3"]
for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

Checking significance for ROC_AUC
(11, 4)
(Original vs 8-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.1
(11, 4)
(Original vs 8-bit): 55.0, p-value: 0.0025167541003031247 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.2
(11, 4)
(Original vs 8-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.3
(11, 4)
(Original vs 8-bit): 34.0, p-value: 0.2532590889119486 The difference is NOT statistically significant.
(Original vs 4-bit): 64.0, p-value: 0.00146484375 The difference is statistically significant.
Checking significance for PR_AUC
(11, 4)
(Origina